# Neurosolit SFT on Colab (Qwen2.5-3B, QLoRA, merged model for the Space)

Runtime > Change runtime type > **T4 GPU**. Add your token as a Colab secret named `HF_TOKEN` (with Notebook access on).

Cell 5 trains, merges and evaluates (never pushes). Read the replies, then run cell 7 to push the **merged** model to a private repo.

In [ ]:
# 1. GPU check + pinned versions
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U "trl==0.19.1" "transformers==4.53.3" "peft==0.21.2" bitsandbytes accelerate datasets huggingface_hub
# The huggingface-hub conflict message about diffusers/gradio is harmless.

In [ ]:
# 2. Upload neurosolit_colab.zip
from google.colab import files
import zipfile, os, shutil

up = files.upload()
name = next(iter(up))
os.makedirs('/content/proj', exist_ok=True)
zipfile.ZipFile(name).extractall('/content/proj')
os.chdir('/content/proj')
os.makedirs('training', exist_ok=True)
os.makedirs('data', exist_ok=True)
for f in list(os.listdir('.')):
    if f == 'train_sft.py':
        shutil.move(f, 'training/train_sft.py')
    elif f.endswith('.jsonl'):
        shutil.move(f, 'data/' + f)
print(sorted(os.listdir('.')))
print(sorted(os.listdir('data')))

In [ ]:
# 3. Sanity check
need = ['run_pipeline.py', 'eval_models.py', 'training/train_sft.py',
        'data/pairs_v4_handwritten.jsonl', 'data/new_reviewed_batch1.jsonl', 'data/new_reviewed_batch2.jsonl', 'data/new_reviewed_batch3.jsonl']
missing = [p for p in need if not os.path.exists(p)]
print('MISSING:' if missing else 'All files present.', missing)

In [ ]:
# 4. Hugging Face login
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get('HF_TOKEN'))
except Exception:
    import getpass
    login(getpass.getpass('HF token: '))

In [ ]:
# 5. Filter data -> train 3B -> merge into a normal model -> eval -> gates. Does NOT push.
!python run_pipeline.py \
  --data data/pairs_v4_handwritten.jsonl \
  --data data/new_reviewed_batch1.jsonl \
  --data data/new_reviewed_batch2.jsonl \
  --data data/new_reviewed_batch3.jsonl \
  --min-rows 60 --epochs 3 --max-length 320 \
  --base-model Qwen/Qwen2.5-3B-Instruct \
  --hf-repo Scadiva/nd-solitude-3b-merged

In [ ]:
# 6. Read more replies yourself, including advice requests (not in the training data), a plain fact, and grief.
!python eval_models.py --model pipeline_out/merged --temperature 0.4 --samples 3 --show 2 \
  --extra-prompt "ways to calm down after a breakup" \
  --extra-prompt "how to convince a person to love you" \
  --extra-prompt "a friend of mine died, how to handle my trauma" \
  --extra-prompt "What is the capital of France?"

In [ ]:
# 7. ONLY if the replies make sense: push the MERGED model to a PRIVATE repo.
!python run_pipeline.py \
  --data data/pairs_v4_handwritten.jsonl \
  --data data/new_reviewed_batch1.jsonl \
  --data data/new_reviewed_batch2.jsonl \
  --data data/new_reviewed_batch3.jsonl \
  --min-rows 60 --max-length 320 --skip-train \
  --base-model Qwen/Qwen2.5-3B-Instruct \
  --hf-repo Scadiva/nd-solitude-3b-merged --push